# Application d’un CNN sur la base MNIST

Dans cette application, nous réalisons un modèle de réseau de neurones convolutif afin de reconnaître des chiffres manuscrits à partir de la base MNIST.

L’objectif est de construire un modèle capable de recevoir une image en entrée et de prédire le chiffre correspondant parmi les classes de 0 à 9.

Le travail est organisé en plusieurs étapes :
1. préparation de l’environnement ;
2. importation des bibliothèques ;
3. chargement de la base MNIST ;
4. visualisation de quelques images ;
5. prétraitement des données ;
6. création du modèle CNN ;
7. entraînement du modèle ;
8. évaluation et analyse des résultats ;
9. sauvegarde du modèle.

## 1. Préparation de l’environnement

Dans cette première étape, nous vérifions la version de TensorFlow utilisée dans Google Colab.  
Nous vérifions également si un GPU est disponible. Le GPU n’est pas obligatoire pour cette application, mais il peut accélérer l’entraînement du modèle.

In [ ]:
# Importation de TensorFlow pour vérifier l'environnement d'exécution
import tensorflow as tf

# Affichage de la version de TensorFlow utilisée
print("Version de TensorFlow :", tf.__version__)

# Vérification de la disponibilité du GPU dans Google Colab
print("GPU disponible :", tf.config.list_physical_devices("GPU"))

## 2. Importation des bibliothèques

Nous importons maintenant les bibliothèques nécessaires pour manipuler les données, afficher les images, construire le réseau CNN et évaluer ses performances.

In [ ]:
# NumPy permet de manipuler les tableaux numériques
import numpy as np

# Matplotlib permet d'afficher les images et les courbes
import matplotlib.pyplot as plt

# Seaborn permet d'afficher une matrice de confusion plus lisible
import seaborn as sns

# os permet de créer un dossier pour enregistrer les figures
import os

# Importation de la base MNIST directement depuis Keras
from tensorflow.keras.datasets import mnist

# Sequential permet de créer un modèle couche par couche
from tensorflow.keras.models import Sequential, load_model

# Importation des couches nécessaires pour construire le CNN
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# to_categorical permet de transformer les labels en vecteurs de classes
from tensorflow.keras.utils import to_categorical

# Outils utilisés pour évaluer le modèle
from sklearn.metrics import confusion_matrix, classification_report

# Création d'un dossier pour sauvegarder les figures générées
os.makedirs("figures", exist_ok=True)

## 3. Chargement de la base MNIST

Nous chargeons la base MNIST. Elle est déjà divisée en deux parties : une partie pour l’entraînement et une partie pour le test.

Les variables `x_train` et `x_test` contiennent les images.  
Les variables `y_train` et `y_test` contiennent les labels, c’est-à-dire les chiffres réels associés aux images.

In [ ]:
# Chargement des images et des labels de la base MNIST
# x_train et y_train sont utilisés pour entraîner le modèle
# x_test et y_test sont utilisés pour tester le modèle après l'entraînement
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Affichage des dimensions pour vérifier que les données sont bien chargées
print("Images d'entraînement :", x_train.shape)
print("Labels d'entraînement :", y_train.shape)
print("Images de test :", x_test.shape)
print("Labels de test :", y_test.shape)

## 4. Visualisation de quelques images

Avant de construire le modèle, nous affichons quelques images de la base MNIST.  
Cette étape permet de mieux comprendre les données utilisées et de vérifier que les images correspondent bien à des chiffres manuscrits.

In [ ]:
# Création d'une figure contenant 10 exemples d'images MNIST
plt.figure(figsize=(10, 4))

# Boucle permettant d'afficher les 10 premières images de l'ensemble d'entraînement
for i in range(10):
    # Création d'une grille de 2 lignes et 5 colonnes
    plt.subplot(2, 5, i + 1)

    # Affichage de l'image en niveaux de gris
    plt.imshow(x_train[i], cmap="gray")

    # Affichage du label réel associé à l'image
    plt.title("Label : " + str(y_train[i]))

    # Suppression des axes pour rendre la figure plus propre
    plt.axis("off")

# Ajustement automatique des espacements
plt.tight_layout()

# Sauvegarde de la figure
plt.savefig("figures/images_mnist.png", dpi=300, bbox_inches="tight")

# Affichage de la figure
plt.show()

## 5. Normalisation des images

Les pixels des images MNIST sont initialement compris entre 0 et 255.  
Pour faciliter l’apprentissage du réseau, nous divisons toutes les valeurs par 255 afin d’obtenir des valeurs comprises entre 0 et 1.

In [ ]:
# Normalisation des images d'entraînement
# Chaque pixel est divisé par 255 pour obtenir une valeur entre 0 et 1
x_train = x_train / 255.0

# Normalisation des images de test avec la même méthode
x_test = x_test / 255.0

## 6. Adaptation de la forme des images

Un réseau CNN attend des images sous la forme :

`nombre d’images, hauteur, largeur, nombre de canaux`

Dans notre cas, les images MNIST ont une taille de 28 × 28 pixels.  
Comme elles sont en niveaux de gris, le nombre de canaux est égal à 1.

In [ ]:
# Redimensionnement des images d'entraînement au format attendu par les couches Conv2D
x_train = x_train.reshape(-1, 28, 28, 1)

# Redimensionnement des images de test de la même manière
x_test = x_test.reshape(-1, 28, 28, 1)

# Vérification des nouvelles dimensions
print("Nouvelle forme de x_train :", x_train.shape)
print("Nouvelle forme de x_test :", x_test.shape)

## 7. Encodage des labels

Les labels sont au départ des chiffres simples comme 0, 1, 2, ..., 9.

Pour utiliser la fonction de perte `categorical_crossentropy`, nous transformons ces labels en vecteurs de taille 10.  
Par exemple, le chiffre 3 devient :

`[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`

In [ ]:
# Transformation des labels d'entraînement en vecteurs de taille 10
y_train_cat = to_categorical(y_train, 10)

# Transformation des labels de test de la même manière
y_test_cat = to_categorical(y_test, 10)

# Affichage d'un exemple pour vérifier la transformation
print("Label initial :", y_train[0])
print("Label après encodage :", y_train_cat[0])

## 8. Création du modèle CNN

Nous construisons maintenant notre modèle CNN.

Le modèle contient :
- deux couches de convolution pour extraire les caractéristiques des images ;
- deux couches de MaxPooling pour réduire la taille des cartes de caractéristiques ;
- une couche Flatten pour transformer les matrices en vecteur ;
- une couche Dense pour la classification ;
- une couche Dropout pour limiter le surapprentissage ;
- une couche de sortie Softmax pour prédire l’un des dix chiffres.

In [ ]:
# Création d'un modèle séquentiel
# Les couches seront ajoutées les unes après les autres
model = Sequential()

# Première couche de convolution
# Elle apprend 32 filtres de taille 3x3 pour détecter des formes simples
model.add(Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)))

# Première couche de MaxPooling
# Elle réduit la taille des cartes de caractéristiques en gardant les informations importantes
model.add(MaxPooling2D((2, 2)))

# Deuxième couche de convolution
# Elle apprend 64 filtres pour détecter des formes plus complexes
model.add(Conv2D(64, (3, 3), activation="relu"))

# Deuxième couche de MaxPooling
# Elle réduit encore la dimension des cartes de caractéristiques
model.add(MaxPooling2D((2, 2)))

# Transformation des cartes de caractéristiques en un seul vecteur
model.add(Flatten())

# Couche entièrement connectée de 128 neurones
# Elle permet de combiner les caractéristiques extraites par les convolutions
model.add(Dense(128, activation="relu"))

# Dropout pour limiter le surapprentissage
# Pendant l'entraînement, 50 % des neurones sont désactivés aléatoirement
model.add(Dropout(0.5))

# Couche de sortie
# Elle contient 10 neurones, un pour chaque chiffre de 0 à 9
# Softmax donne une probabilité pour chaque classe
model.add(Dense(10, activation="softmax"))

## 9. Compilation du modèle

Avant l’entraînement, nous devons compiler le modèle.

Nous utilisons :
- l’optimiseur `Adam` pour ajuster les poids du réseau ;
- la fonction de perte `categorical_crossentropy`, adaptée à une classification multi-classes ;
- la métrique `accuracy` pour suivre le taux de bonnes prédictions.

In [ ]:
# Compilation du modèle
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Affichage du résumé du modèle pour vérifier l'architecture
model.summary()

## 10. Entraînement du modèle

Nous entraînons le modèle sur les images d’entraînement.

Dans cette application :
- `epochs=5` signifie que le modèle parcourt 5 fois toute la base d’entraînement ;
- `batch_size=128` signifie que les images sont traitées par groupes de 128 ;
- `validation_split=0.1` signifie que 10 % des données d’entraînement sont réservées pour la validation.

In [ ]:
# Entraînement du modèle CNN
history = model.fit(
    x_train,
    y_train_cat,
    epochs=5,
    batch_size=128,
    validation_split=0.1
)

## 11. Évaluation du modèle

Après l’entraînement, nous évaluons le modèle sur les images de test.

Ces images n’ont pas été utilisées pendant l’apprentissage.  
Elles permettent donc de vérifier si le modèle est capable de généraliser sur de nouvelles données.

In [ ]:
# Évaluation du modèle sur les données de test
test_loss, test_accuracy = model.evaluate(x_test, y_test_cat)

# Affichage des résultats obtenus
print("Erreur sur les données de test :", test_loss)
print("Précision sur les données de test :", test_accuracy)

## 12. Courbe de précision

Nous traçons maintenant la courbe de précision afin d’observer l’évolution des performances du modèle pendant l’entraînement.

In [ ]:
# Création de la courbe de précision
plt.figure(figsize=(8, 5))

# Précision obtenue sur les données d'entraînement
plt.plot(history.history["accuracy"], label="Précision entraînement")

# Précision obtenue sur les données de validation
plt.plot(history.history["val_accuracy"], label="Précision validation")

# Ajout des informations de la figure
plt.xlabel("Époques")
plt.ylabel("Précision")
plt.title("Évolution de la précision")
plt.legend()
plt.grid()

# Sauvegarde de la figure
plt.savefig("figures/courbe_precision.png", dpi=300, bbox_inches="tight")

# Affichage de la figure
plt.show()

## 13. Courbe de perte

Nous traçons aussi la courbe de perte afin de suivre l’évolution de l’erreur du modèle pendant l’apprentissage.

In [ ]:
# Création de la courbe de perte
plt.figure(figsize=(8, 5))

# Erreur obtenue sur les données d'entraînement
plt.plot(history.history["loss"], label="Erreur entraînement")

# Erreur obtenue sur les données de validation
plt.plot(history.history["val_loss"], label="Erreur validation")

# Ajout des informations de la figure
plt.xlabel("Époques")
plt.ylabel("Erreur")
plt.title("Évolution de la fonction de perte")
plt.legend()
plt.grid()

# Sauvegarde de la figure pour le rapport
plt.savefig("figures/courbe_erreur.png", dpi=300, bbox_inches="tight")

# Affichage de la figure
plt.show()

## 14. Prédictions du modèle

Après l’évaluation, nous utilisons le modèle pour prédire les classes des images de test.

Le modèle donne d’abord une probabilité pour chaque classe.  
Ensuite, nous utilisons `np.argmax` pour récupérer la classe ayant la probabilité la plus élevée.

In [ ]:
# Prédiction des probabilités pour chaque image de test
y_pred = model.predict(x_test)

# Conversion des probabilités en classes prédites
# np.argmax récupère l'indice de la probabilité la plus élevée
y_pred_classes = np.argmax(y_pred, axis=1)

## 15. Affichage de quelques prédictions

Nous affichons quelques images de test avec la classe prédite par le modèle et le vrai label.  
Cela permet de vérifier visuellement les résultats.

In [ ]:
# Affichage de quelques images de test avec la classe prédite par le modèle
plt.figure(figsize=(10, 4))

for i in range(10):
    # Création de la grille d'affichage
    plt.subplot(2, 5, i + 1)

    # Affichage de l'image de test
    plt.imshow(x_test[i].reshape(28, 28), cmap="gray")

    # Affichage de la prédiction et du vrai label
    plt.title("Prédit : " + str(y_pred_classes[i]) + " / Réel : " + str(y_test[i]))

    # Suppression des axes
    plt.axis("off")

# Ajustement automatique des espacements
plt.tight_layout()

# Sauvegarde de la figure pour le rapport
plt.savefig("figures/predictions_mnist.png", dpi=300, bbox_inches="tight")

# Affichage de la figure
plt.show()

## 16. Matrice de confusion

La matrice de confusion permet d’analyser les erreurs du modèle.

Elle montre les classes réelles en ligne et les classes prédites en colonne.  
Les valeurs situées sur la diagonale principale correspondent aux prédictions correctes.

In [ ]:
# Calcul de la matrice de confusion à partir des vrais labels et des classes prédites
cm = confusion_matrix(y_test, y_pred_classes)

# Affichage de la matrice de confusion
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

# Ajout des titres et des axes
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.title("Matrice de confusion")

# Sauvegarde de la figure pour le rapport
plt.savefig("figures/matrice_confusion.png", dpi=300, bbox_inches="tight")

# Affichage de la figure
plt.show()

## 17. Rapport de classification

Pour compléter l’analyse, nous affichons un rapport de classification.  
Il contient la précision, le rappel et le F1-score pour chaque chiffre.

In [ ]:
# Affichage d'un rapport détaillé : précision, rappel et F1-score pour chaque classe
print(classification_report(y_test, y_pred_classes))

## 18. Sauvegarde du modèle

Enfin, nous sauvegardons le modèle au format natif Keras.  
Cela permet de réutiliser le modèle plus tard sans refaire l’entraînement.

In [ ]:
# Sauvegarde locale du modèle au format recommandé par Keras
model.save("modele_cnn_mnist.keras")

## 19. Vérification du rechargement du modèle

Pour vérifier que la sauvegarde fonctionne correctement, nous rechargeons le modèle sauvegardé.

In [ ]:
# Rechargement du modèle sauvegardé
model_recharge = load_model("modele_cnn_mnist.keras")

# Message de confirmation
print("Modèle rechargé avec succès.")

In [ ]:
# Connexion à Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Sauvegarde du modèle dans Google Drive
model.save("/content/drive/MyDrive/modele_cnn_mnist.keras")